In [ ]:
%cd ../src

In [ ]:
import logging

import polars as pl

from brokers.kite import KiteHistorical, KiteLogin, fetch_instruments
from conf import KITE_CREDENTIALS_PATH
from config.brokers import KiteConfig
from utils import setup_logger

logger = logging.getLogger()
setup_logger()

# Kite Login Check

In [ ]:
kite_conn = KiteLogin(credentials_path=KITE_CREDENTIALS_PATH).auto_login()

In [ ]:
kite_conn.orders()

# Instruments List Check

In [ ]:
ins_df = fetch_instruments(kite=kite_conn, exchange=kite_conn.EXCHANGE_NSE)

In [ ]:
ins_df.sample(10)

In [ ]:
ins_df.filter(pl.col("symbol").is_in(["ITC", "PROTEAN", "SBIN"])).select(
    "symbol", "instrument_token"
)

# Historical Data

In [ ]:
kite_hist = KiteHistorical(
    kite_conn,
    instruments_df=ins_df.filter(
        pl.col("symbol").is_in(["ITC", "PROTEAN", "SBIN"])
    ).select("symbol", "instrument_token"),
    config=KiteConfig,
)

In [ ]:
_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=False
)
_fl_tbl_name = KiteConfig.get_db_tbl(
    data_type=KiteConfig.TYP_STOCKS, frequency="day", failed_tbl=True
)

In [ ]:
kite_hist.get_historical_data(
    start_date="2026-01-01 00:00:00",
    end_date="2026-11-21 00:00:00",
    frequency="day",
    db_conn=KiteConfig.DB_CONN,
    insert_table_name=_tbl_name,
    failed_table_name=_fl_tbl_name,
)

In [ ]:
df = pl.read_database_uri(
    query=f"SELECT * from {_tbl_name}",
    uri=KiteConfig.DB_CONN,
    engine="connectorx"
)

In [ ]:
df.head()